In [37]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
import os
import numpy as np

import pandas as pd
from astropy.io.votable import parse
from tqdm import tqdm

import platosim.utilities as ut

In [ ]:
def passbandConversionG2P(mag, BP_RP, inverse=False, camera='normal'):

    """Conversion from Gaia G magnitude to the PLATO passband.
    
    The calibration relation is derived in PLATO-UPD-SCI-TN-0019, Sect. 6.
    NOTE only valid for (4000K < Teff < 15000K), hence, not for M-dwarfs.

    Parameters
    ----------
    mag : float, narray
        Gaia mean dereddened G magnitude of star(s).
    BP_RP : float narray
        Gaia dereddened color of star(s).

    Return
    ------
    P : float, narray
        The PLATO passband magnitude of star(s).
    """

    # Define coefficient to transform from G to P
    
    if camera == 'normal':
        coeff = [-0.3613390, 0.0632494, 0.0301607, -0.0163962, 0.0027984, -0.0001679]
    elif camera == 'fast_blue':
        coeff = [-0.1386193, 0.1103836, 0.0582385, -0.0144120, 0.0006554, 0.0000251]
    elif camera == 'fast_red':
        coeff = [-0.6795686, 0.0539941, 0.0331913, -0.0123407, 0.0019006, -0.0001174]

    color = np.sum([coeff[i-1] * BP_RP**i for i in range(1,7)], axis=0)
    
    # From G to P (or P to G if inverse=True)
    
    if inverse:
        return mag - color 
    else:
        return mag + color

## PIC 1.1.0

## PIC 2.0.0

In [20]:
field = 'LOPN1'
path  = '/lhome/nicholas/software/pLOPN1PIC2.0.0.1-t' 

### Targets

In [21]:
# Load targets
tfile = f'p{field}PICtarget2.0.0.1-t.vot'
votable = parse(f'{path}/{tfile}')
df0 = ut.votable2pandas(votable)

# Write relevant columns to df
df = pd.DataFrame()
df['PIC']   = df0.PICid
df['ra']    = df0.RAdeg
df['dec']   = df0.DEdeg
df['Pmag']  = df0.PlatoMagNCAM
df['PBmag'] = df0.PlatoMagFCAMb
df['PRmag'] = df0.PlatoMagFCAMr
df['Teff']  = df0.Teff
df['R']     = df0.Radius
df['M']     = df0.Mass
df['ncams'] = df0.BOLnCameraObs

# Remove NaNs
df = df.dropna()

# Remove stellar duplicates (if any)
df = df.drop_duplicates(subset=['PIC'])
        
# Save to feather files
df = df.reset_index(drop=True)
df.to_feather(f'PIC200_{field}_targets.ftr')

Loading PIC targets


### Contaminants

In [72]:
# File is huge hence read only one column at the time
cfile = f'p{field}PICcontaminant2001t.csv'

# Create data frame
dc  = pd.DataFrame()
dc['PIC']   = pd.read_csv(f'{path}/{cfile}', usecols=['PICcontaminantId'])
dc['ra']    = pd.read_csv(f'{path}/{cfile}', usecols=['RAdeg'])
dc['dec']   = pd.read_csv(f'{path}/{cfile}', usecols=['DEdeg'])
dc['Gmag']  = pd.read_csv(f'{path}/{cfile}', usecols=['Gmag'])
dc['BPmag'] = pd.read_csv(f'{path}/{cfile}', usecols=['BPmag'])
dc['RPmag'] = pd.read_csv(f'{path}/{cfile}', usecols=['RPmag'])

# Remove NaNs
dc = dc.dropna()

In [73]:
# Use Gaia colours to convert to PLATO bandpass
dc['Pmag']  = ut.passbandConversionG2P(dc.Gmag, dc.BPmag-dc.RPmag)
dc['PBmag'] = ut.passbandConversionG2P(dc.Gmag, dc.BPmag-dc.RPmag, camera='fast_blue')
dc['PRmag'] = ut.passbandConversionG2P(dc.Gmag, dc.BPmag-dc.RPmag, camera='fast_red')

# Remove Gaia filters again
dc = dc.drop(columns=['Gmag', 'BPmag', 'RPmag'])

In [81]:
# Fetch all contaminants within 45 arcsec from target
for i in tqdm(range(df.shape[0]), bar_format=ut.tqdmBar()):
    
    # Select target star
    df_i = df.iloc[i]
    
    # Fetch smaller region around target
    x = 45/3600.
    dc_i = dc[(dc.ra  > df_i.ra  - x) & (dc.ra  < df_i.ra  + x) &
              (dc.dec > df_i.dec - x) & (dc.dec < df_i.dec + x)]
    
    # Remove target star if present
    dc_i = dc_i.drop(dc_i[dc_i.PIC == df_i.PIC].index)
    
    # Find radial distance [arcsec]
    dc_i['dis'] = ut.radialDistance(df_i.ra, df_i.dec,
                                    dc_i.ra.to_numpy(), 
                                    dc_i.dec.to_numpy()) * 3600.
    dc_i = dc_i.sort_values(by=['dis'])
            
    # Save to a new df
    if i == 0:
        dc0 = dc_i
    else:
        dc0 = pd.concat([dc0, dc_i])
        
# Save to feather files
dc0 = dc0.reset_index(drop=True)
dc0.to_feather(f'PIC200_{field}_contaminants.ftr')

 74%|████████████████████████████████████▊             | 128927/175325 [12:17:34<4:25:26,  2.91it/s] 


KeyboardInterrupt: 